# Phase 1C: Phase 1B augmentation on timestamp-fixed v2

Only dataset/v2_timestamp_fixed supplies model inputs and fitted preprocessing. All new augmentation artifacts go to augmentation_experiment/v2_timestamp_fixed/. Four variants retain seed 42 and the original relevance weighting/noise policy. Independent source audits and acceptance gates remain active. Original v1 outputs are immutable.

Weighted complete-window sampling is a fallback, not full SMOGN; no interpolation. No training, Optuna, waveforms or fusion. Methods and references are saved in research_adaptation_notes.md.


In [1]:
from pathlib import Path
import json, hashlib, math
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT=Path.cwd().resolve()
if ROOT.name!="bilstm new updated":ROOT=ROOT/"bilstm new updated"
DATA=ROOT/"dataset"/"v2_timestamp_fixed"; OUT=ROOT/"augmentation_experiment"/"v2_timestamp_fixed"
OUT.mkdir(parents=True,exist_ok=True)
SEED=42
NS_PER_SECOND=1_000_000_000
SECONDS_PER_DAY=86_400
def digest(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):h.update(block)
    return h.hexdigest()
def tensor_hash(t):
    a=t.detach().cpu().contiguous().numpy()
    h=hashlib.sha256()
    h.update(str(a.dtype).encode());h.update(str(a.shape).encode());h.update(a.tobytes())
    return h.hexdigest()
def write_json(path,obj):
    path=Path(path);tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,allow_nan=False),encoding="utf-8");tmp.replace(path)
# Hash already registered clean artifacts without loading their numerical contents.
original_registry=json.loads((DATA/"catalog_artifact_hashes.json").read_text())
protected_before={name:digest(DATA/name) for name in original_registry["artifacts"]}
assert protected_before==original_registry["artifacts"],"Source artifact hash mismatch."
protected_before["catalog_artifact_hashes.json"]=digest(DATA/"catalog_artifact_hashes.json")
schema=json.loads((DATA/"feature_schema.json").read_text())
scaling=json.loads((DATA/"catalog_train_scalers.json").read_text())
prior_validation=json.loads((DATA/"dataset_validation.json").read_text())
assert prior_validation.get("training_ready") is True
assert not any(v["status"]=="FAIL" for v in prior_validation["checks"].values())
archived_aug=ROOT/"augmentation_experiment"
archived_registry=json.loads((archived_aug/"artifact_hashes.json").read_text())
archived_before={name:digest(archived_aug/name) for name in archived_registry["outputs"]}
assert archived_before==archived_registry["outputs"]
archived_registry_hash=digest(archived_aug/"artifact_hashes.json")
manifest=pd.read_csv(DATA/"split_manifest.csv",keep_default_na=False)
events=pd.read_csv(DATA/"catalog_events.csv",keep_default_na=False)
clean=torch.load(DATA/"bilstm_catalog_seq20.pt",map_location="cpu",weights_only=True)
original_tensor_hashes={k:tensor_hash(v) for k,v in clean.items()}
assert clean["X_train"].shape==(4973,20,10)
assert schema["run_id"]==scaling["run_id"]==prior_validation["run_id"]
names=[f["name"] for f in schema["ordered_features"]]
assert names==scaling["feature_order"]
coord_cols=[names.index("historical_latitude_deg"),names.index("historical_longitude_deg")]
assert coord_cols==[2,3]
mean=np.array(scaling["scaler"]["mean"])
scale=np.array(scaling["scaler"]["scale"])
train_manifest=manifest.loc[manifest.split.eq("train")].reset_index(drop=True)
X=clean["X_train"].numpy();y=clean["y_reg_train"].numpy();binary=clean["y_binary_train"].numpy()
N=len(y)
assert len(train_manifest)==N and np.allclose(y,train_manifest.target_magnitude,atol=1e-6)
assert np.array_equal(binary,(y>=4.5).astype(np.int64))
input_ids=np.array([json.loads(v) for v in train_manifest.input_event_ids])
event_lookup=dict(zip(events.source_id,events.event_index.astype(int)))
input_event_idx=np.array([[event_lookup[v] for v in row] for row in input_ids])
target_idx=train_manifest.target_event_index.to_numpy(dtype=int)
# Convert resolution explicitly: pandas' native integer datetime units need not be nanoseconds.
event_time=pd.to_datetime(events.time,utc=True)
native_unit=str(event_time.dtype)
time_ns=event_time.dt.as_unit("ns").astype("int64").to_numpy()
actual_gaps=(event_time.iloc[target_idx].reset_index(drop=True)-
             event_time.iloc[input_event_idx[:,-1]].reset_index(drop=True)).dt.total_seconds().to_numpy()
actual_input_before=(time_ns[input_event_idx]<time_ns[target_idx,None]).all()
assert actual_input_before and (np.diff(time_ns[input_event_idx],axis=1)>0).all()
train_ids=set(train_manifest.target_event_id)
assert train_ids.isdisjoint(set(manifest.loc[manifest.split.ne("train"),"target_event_id"]))
CONFIG={"version":"phase1b_v2_timestamp_fixed","source_run_id":schema["run_id"],"seed":SEED,"preview_epoch":0,
 "variants":{"A":"clean","B":"coordinate_noise","C":"weighted_complete_windows","D":"weighted_complete_windows_plus_coordinate_noise"},
 "sampler":{"replacement":True,"epoch_size":N,"sort_parent_indices":True,
            "weight_formula":"1 + 0.5 * clip((training_target_ML - 4.0) / 1.5, 0, 1)",
            "max_weight":1.5,"relevance_start":4.0,"relevance_saturation":5.5,
            "force_class_balance":False,"neighbors":None,"interpolation":None},
 "noise":{"enabled_variants":["B","D"],"sigma_standardized":0.01,"eligible_columns":coord_cols,
          "eligible_features":[names[i] for i in coord_cols],"seed":SEED,
          "scope":"training only; deterministic per historical event and epoch, shared across B/D",
          "physical_bounds":[[-90,90],[-180,180]],"targets_perturbed":False},
 "source_feature_check_policy":"Reject training-ready use if source schema or provenance fails independent checks",
 "selection_policy":"Settings fixed from user guidance and training targets only; no validation/test tuning"}
RUN_ID=hashlib.sha256((json.dumps(CONFIG,sort_keys=True)+protected_before["bilstm_catalog_seq20.pt"]).encode()).hexdigest()[:16]
CONFIG["run_id"]=RUN_ID
write_json(OUT/"augmentation_config.json",CONFIG)
print("Loaded clean catalog tensors once; source hashes match; source run",schema["run_id"])


Loaded clean catalog tensors once; source hashes match; source run 945fdfac7b47cfb0


## Independent source checks

These calculations audit saved feature semantics using saved catalog provenance; they do not regenerate the dataset or refit preprocessing. Rates are checked using timezone-aware Timedelta arithmetic rather than the original implementation's integer units. All feature audits use training windows only. Validation/test contents are hashed for immutability, not used as neighbors or to choose settings.


In [2]:
raw=X.astype(np.float64)*scale+mean
# Independent time-index rolling count, including the event at the right endpoint.
expected_event_rate=pd.Series(np.ones(len(events)),index=pd.DatetimeIndex(event_time)).rolling("30D",closed="right").sum().to_numpy()/30.
expected_rate=expected_event_rate[input_event_idx]
rate_col=names.index("event_rate_30d")
rate_bad=~np.isclose(raw[...,rate_col],expected_rate,atol=2e-5,rtol=2e-6)
saved_gaps=train_manifest.input_to_target_gap_seconds.to_numpy(dtype=float)
gap_bad=~np.isclose(saved_gaps,actual_gaps,atol=1e-6,rtol=1e-9)
# Confirm unaffected core features and rolling relations against the retained event records.
mag=events.magnitude.astype(float);depth=events.depth_km.astype(float)
expected_columns={
 0:mag.to_numpy()[input_event_idx],1:depth.to_numpy()[input_event_idx],
 2:events.latitude_deg.astype(float).to_numpy()[input_event_idx],
 3:events.longitude_deg.astype(float).to_numpy()[input_event_idx],
 4:event_time.diff().dt.total_seconds().to_numpy()[input_event_idx]/SECONDS_PER_DAY,
 6:mag.rolling(20,min_periods=1).mean().to_numpy()[input_event_idx],
 7:mag.rolling(20,min_periods=1).std(ddof=0).to_numpy()[input_event_idx],
 8:depth.rolling(20,min_periods=1).mean().to_numpy()[input_event_idx],
 9:depth.rolling(20,min_periods=1).std(ddof=0).to_numpy()[input_event_idx]}
column_failures={}
for col,expected in expected_columns.items():
    missing=np.isnan(expected)
    expected=np.where(missing,scaling["imputer"]["statistics"][col],expected)
    column_failures[names[col]]=int((~np.isclose(raw[...,col],expected,atol=3e-5,rtol=3e-6)).sum())
rows=[]
for i,j in np.argwhere(rate_bad)[:8]:
    ei=int(input_event_idx[i,j])
    rows.append({"training_parent_row":int(i),"input_step":int(j),"source_id":str(input_ids[i,j]),
                 "event_time":str(event_time.iloc[ei]),"saved_rate_per_day":float(raw[i,j,rate_col]),
                 "expected_rate_per_day":float(expected_rate[i,j])})
# Diagnose unit mismatch from saved preprocessing source, without executing it.
source_nb=json.loads((ROOT/"phase1_bilstm_csv_preprocessing.ipynb").read_text(encoding="utf-8"))
relevant_lines=[line for c in source_nb["cells"] if c["cell_type"]=="code"
                for line in "".join(c["source"]).splitlines()
                if any(term in line for term in ['astype("int64")','Timedelta(days=30).value','/1e9'])]
source_ok=prior_validation.get("training_ready") is True and not rate_bad.any() and not gap_bad.any() and not any(column_failures.values())
source_audit={"status":"PASS" if source_ok else "FAIL","training_ready":bool(source_ok),
 "native_parsed_timestamp_dtype":native_unit,"explicit_audit_timestamp_unit":"ns",
 "incorrect_rate_feature_cells":int(rate_bad.sum()),"training_windows_with_incorrect_rate":int(rate_bad.any(axis=1).sum()),
 "incorrect_training_gap_records":int(gap_bad.sum()),"other_feature_constraint_violations":column_failures,
 "gap_expected_divided_by_saved_min":float((actual_gaps/saved_gaps).min()),
 "gap_expected_divided_by_saved_max":float((actual_gaps/saved_gaps).max()),
 "strict_origin_chronology":"PASS" if actual_input_before else "FAIL",
 "rate_examples":rows,"source_notebook_relevant_lines":relevant_lines,
 "cause":"V2 explicitly normalizes integer timestamps to ns; independent rate, gap and all-feature checks verify the correction." if source_ok else "V2 source consistency failed; inspect mismatch evidence.",
 "correction_policy":"Use only dataset/v2_timestamp_fixed; original v1 datasets and scalers remain unchanged.",
 "prior_validation_limit":"Previous prefix checks reused the same feature logic and could not detect a shared time-unit error.",
 "operational_feature_availability":"UNKNOWN"}
write_json(OUT/"source_feature_audit.json",source_audit)
print(json.dumps({k:source_audit[k] for k in ["status","native_parsed_timestamp_dtype",
 "incorrect_rate_feature_cells","training_windows_with_incorrect_rate","incorrect_training_gap_records",
 "gap_expected_divided_by_saved_min","gap_expected_divided_by_saved_max"]}))


{"status": "PASS", "native_parsed_timestamp_dtype": "datetime64[us, UTC]", "incorrect_rate_feature_cells": 0, "training_windows_with_incorrect_rate": 0, "incorrect_training_gap_records": 0, "gap_expected_divided_by_saved_min": 1.0, "gap_expected_divided_by_saved_max": 1.0}


## Reproducible conservative sampler and constrained noise

Weights are `1 + 0.5×clip((ML−4)/1.5,0,1)`, based only on training regression labels: a bounded relevance heuristic, not a learned or paper-reproduced relevance function. Each epoch has the original number of draws, with replacement for C/D. Draws are sorted by original training-row order, preserving chronological order both within windows and across the sampled sequence. Sampling changes exposure, not features/labels or target identity; no 50/50 quota.

Noise uses seed 42, σ=0.01, historical coordinates only. An event receives the same perturbation wherever it recurs within an epoch, including across B/D; it is independent of target labels. Physical bounds are enforced. Perturbed observations are marked SYNTHETIC_FEATURE_PERTURBATION, never new earthquakes. Diagnostic previews may be audited even when the original data fails; the training consumer refuses them.


In [3]:
weights=1.0+0.5*np.clip((y.astype(np.float64)-4.0)/1.5,0,1)
probabilities=weights/weights.sum()
unique_history_ids=np.unique(input_ids)
history_id_to_row={v:i for i,v in enumerate(unique_history_ids)}
history_rows=np.array([[history_id_to_row[v] for v in row] for row in input_ids])
def draw_parent_indices(variant,epoch=0,*,split_name="train",sampling_seed=SEED):
    if split_name!="train":raise ValueError("Sampling is restricted to training parents.")
    if variant not in CONFIG["variants"]:raise ValueError("Unknown variant.")
    if variant in ("A","B"):return np.arange(N,dtype=np.int64)
    rng=np.random.default_rng(np.random.SeedSequence([sampling_seed,int(epoch),17]))
    return np.sort(rng.choice(N,size=N,replace=True,p=probabilities)).astype(np.int64)
def coordinate_noise(clean_inputs,parent_rows,*,epoch=0,enabled=True,sigma=0.01,split_name="train"):
    if split_name!="train":raise ValueError("Noise is restricted to training inputs.")
    if sigma<0:raise ValueError("sigma must be nonnegative")
    out=np.array(clean_inputs,copy=True)
    if not enabled or sigma==0:return out,0
    rng=np.random.default_rng(np.random.SeedSequence([SEED,int(epoch),29]))
    noise_table=rng.normal(0,sigma,size=(len(unique_history_ids),2))
    noise=noise_table[history_rows[parent_rows]]
    clipped=0
    for pos,(col,lo,hi) in enumerate([(2,-90.,90.),(3,-180.,180.)]):
        before=out[...,col].astype(np.float64)+noise[...,pos]
        lower=(lo-mean[col])/scale[col];upper=(hi-mean[col])/scale[col]
        clipped+=int(((before<lower)|(before>upper)).sum())
        out[...,col]=np.clip(before,lower,upper)
    return out,clipped
def diagnostic_variant(variant,epoch=0):
    parents=draw_parent_indices(variant,epoch)
    x,clipped=coordinate_noise(X[parents],parents,epoch=epoch,enabled=variant in ("B","D"),
                              sigma=CONFIG["noise"]["sigma_standardized"])
    return parents,x,y[parents].copy(),binary[parents].copy(),clipped
def prepare_training_epoch(variant,epoch=0):
    if not source_ok:
        raise RuntimeError("Training-ready export blocked: original rate/gap semantics failed independent checks. See source_feature_audit.json.")
    return diagnostic_variant(variant,epoch)[:4]

preview={v:diagnostic_variant(v) for v in CONFIG["variants"]}
plan={f"parent_rows_{v}":preview[v][0] for v in preview}
plan["weights"]=weights
np.savez_compressed(OUT/"sampling_plan.npz",**plan)
write_json(OUT/"sampler_manifest.json",{
 "run_id":RUN_ID,"training_ready":bool(source_ok),"status":"ACCEPTED" if source_ok else "DIAGNOSTIC_ONLY_SOURCE_INVALID",
 "source_dataset":"../../dataset/v2_timestamp_fixed/bilstm_catalog_seq20.pt","source_sha256":protected_before["bilstm_catalog_seq20.pt"],
 "parameter_file":"augmentation_config.json","preview_epoch":0,"preview_plan":"sampling_plan.npz",
 "epoch_implementation":"draw_parent_indices, coordinate_noise and prepare_training_epoch in ../../phase1b_imbalance_noise_preprocessing.ipynb",
 "numpy_version":np.__version__,"bit_generator":"PCG64 via default_rng/SeedSequence",
 "historical_event_noise_indexing":"Sorted unique training input source IDs from original split manifest; IDs never added as model features.",
 "new_earthquake_records":0,"interpolated_examples":0,
 "consumer_gate":"prepare_training_epoch refuses use when source validation fails; diagnostic previews are not training approval."})
provenance=[];comparison=[];variant_checks={}
unchanged_cols=[i for i in range(10) if i not in coord_cols]
bin_edges=[-np.inf,3.,4.,4.5,5.,np.inf]
bin_names=["<3","3–<4","4–<4.5","4.5–<5",">=5"]
for v,(parents,x,yr,yb,clipped) in preview.items():
    duplicated=pd.Series(parents).duplicated().to_numpy()
    for draw,parent in enumerate(parents):
        provenance.append({"variant":v,"epoch":0,"draw_index":draw,"parent_training_row":int(parent),
          "parent_sample_index":int(train_manifest.sample_index.iloc[parent]),
          "parent_target_event_id":train_manifest.target_event_id.iloc[parent],
          "parent_split":"train","record_role":"SYNTHETIC_FEATURE_PERTURBATION" if v in ("B","D") else "EXISTING_WINDOW_DRAW",
          "repeated_parent_draw":bool(duplicated[draw]),"interpolation_method":"NONE",
          "noise_sigma":0.01 if v in ("B","D") else 0.0,"seed":SEED,
          "training_ready":bool(source_ok),"original_manifest":"../../dataset/v2_timestamp_fixed/split_manifest.csv"})
    bounds_violations=0
    for col,lo,hi in [(2,-90.,90.),(3,-180.,180.)]:
        unscaled=x[...,col].astype(np.float64)*scale[col]+mean[col]
        bounds_violations+=int(((unscaled<lo-1e-5)|(unscaled>hi+1e-5)).sum())
    constraints={
      "shape_and_finite":bool(x.shape==(N,20,10) and np.isfinite(x).all()),
      "labels_equal_parent":bool(np.array_equal(yr,y[parents]) and np.array_equal(yb,binary[parents])),
      "binary_target_consistent":bool(np.array_equal(yb,(yr>=4.5).astype(np.int64))),
      "parents_training_only":bool(np.all((parents>=0)&(parents<N))),
      "parent_order_chronological":bool((np.diff(parents)>=0).all()),
      "unperturbed_features_equal_parent":bool(np.array_equal(x[...,unchanged_cols],X[parents][...,unchanged_cols])),
      "coordinates_within_bounds":bounds_violations==0,
      "source_feature_constraints":bool(source_ok)}
    assert all(val for key,val in constraints.items() if key!="source_feature_constraints")
    inherited_bad_windows=rate_bad.any(axis=1)[parents]
    counts=np.histogram(yr,bins=bin_edges)[0]
    comparison.append({"variant":v,"method":CONFIG["variants"][v],"training_draws":N,
       "unique_parent_windows":int(len(np.unique(parents))),"repeat_draws":int(duplicated.sum()),
       "synthetic_feature_examples":N if v in ("B","D") else 0,"interpolated_examples":0,"new_event_records":0,
       "shape":list(x.shape),"magnitude_mean":float(yr.mean()),"magnitude_std":float(yr.std()),
       "magnitude_min":float(yr.min()),"magnitude_max":float(yr.max()),
       "magnitude_bin_counts":dict(zip(bin_names,counts.tolist())),
       "M_ge_4_5_count":int(yb.sum()),"M_ge_4_5_fraction":float(yb.mean()),
       "finite":bool(np.isfinite(x).all()),"coordinate_bound_violations":bounds_violations,
       "coordinate_values_clipped":clipped,"new_feature_constraint_violations":0,
       "inherited_invalid_rate_windows":int(inherited_bad_windows.sum()),
       "inherited_invalid_gap_records":int(gap_bad[parents].sum()),
       "X_content_sha256":tensor_hash(torch.from_numpy(x)),
       "regression_target_sha256":tensor_hash(torch.from_numpy(yr)),
       "binary_target_sha256":tensor_hash(torch.from_numpy(yb)),
       "status":"ACCEPTED" if source_ok else "REJECTED_SOURCE_FEATURE_CONSTRAINTS"})
    variant_checks[v]=constraints
pd.DataFrame(provenance).to_csv(OUT/"parent_provenance.csv",index=False)
write_json(OUT/"distribution_comparison.json",comparison)
flat=[{**r,"shape":str(r["shape"]),"magnitude_bin_counts":json.dumps(r["magnitude_bin_counts"])} for r in comparison]
pd.DataFrame(flat).to_csv(OUT/"distribution_comparison.csv",index=False)
print(pd.DataFrame(comparison)[["variant","training_draws","unique_parent_windows","repeat_draws","M_ge_4_5_count","M_ge_4_5_fraction","status"]].to_string(index=False))


variant  training_draws  unique_parent_windows  repeat_draws  M_ge_4_5_count  M_ge_4_5_fraction   status
      A            4973                   4973             0            1553           0.312286 ACCEPTED
      B            4973                   4973             0            1553           0.312286 ACCEPTED
      C            4973                   3114          1859            1822           0.366378 ACCEPTED
      D            4973                   3114          1859            1822           0.366378 ACCEPTED


## Leakage, replay and acceptance checks

The weighted preview is one reproducible epoch, not new independent observations. C/D use identical parent draws; B/D apply identical event-level noise. All original file/tensor hashes must remain unchanged. A structural source failure rejects every training-ready variant, even if the sampler/noise operations introduce no additional constraint violation. No performance comparison is possible without future model training.


In [4]:
# Replay persisted sampler arrays and noise implementation; all outputs must reproduce exactly.
with np.load(OUT/"sampling_plan.npz",allow_pickle=False) as saved_plan:
    assert np.array_equal(saved_plan["weights"],weights)
    for v in preview:
        regenerated=diagnostic_variant(v)
        assert np.array_equal(saved_plan[f"parent_rows_{v}"],regenerated[0])
        assert all(np.array_equal(a,b) for a,b in zip(preview[v][:4],regenerated[:4]))
assert np.array_equal(preview["A"][0],preview["B"][0])
assert np.array_equal(preview["C"][0],preview["D"][0])
assert np.array_equal(preview["B"][1][preview["D"][0]],preview["D"][1])
assert np.array_equal(preview["A"][1],X)
assert np.array_equal(preview["C"][1],X[preview["C"][0]])
assert not np.array_equal(preview["B"][1][...,coord_cols],X[...,coord_cols])
# Noise switch, alternative epoch and rejection of validation/test augmentation.
off,_=coordinate_noise(X,np.arange(N),enabled=False)
assert np.array_equal(off,X)
assert not np.array_equal(draw_parent_indices("C",0),draw_parent_indices("C",1))
rejections=[]
for split_name in ["val","test"]:
    for operation in ["sampling","noise"]:
        try:
            if operation=="sampling":draw_parent_indices("C",split_name=split_name)
            else:coordinate_noise(X[:1],np.array([0]),split_name=split_name)
        except ValueError:rejections.append(split_name+"_"+operation)
assert len(rejections)==4
if not source_ok:
    try:prepare_training_epoch("A")
    except RuntimeError:consumer_blocked=True
    else:consumer_blocked=False
    assert consumer_blocked
else:
    consumer_blocked=False
    for variant in preview:
        accepted=prepare_training_epoch(variant,0)
        assert all(np.array_equal(a,b) for a,b in zip(accepted,preview[variant][:4]))
# Verify persisted provenance joins training parents, without fabricating new target event IDs.
saved_prov=pd.read_csv(OUT/"parent_provenance.csv",keep_default_na=False)
assert len(saved_prov)==4*N
for v in preview:
    p=saved_prov.loc[saved_prov.variant.eq(v)].reset_index(drop=True)
    parents=p.parent_training_row.to_numpy(dtype=int)
    assert np.array_equal(parents,preview[v][0])
    assert np.array_equal(p.parent_target_event_id,train_manifest.target_event_id.iloc[parents].to_numpy())
    assert p.parent_split.eq("train").all()
    assert p.interpolation_method.eq("NONE").all()
protected_after={name:digest(DATA/name) for name in protected_before}
assert protected_before==protected_after
assert original_tensor_hashes=={k:tensor_hash(v) for k,v in clean.items()}
csv_hash_matches=digest(ROOT.parent/"dataset_2.csv")==original_registry["source_sha256"]
assert csv_hash_matches
assert archived_before=={name:digest(archived_aug/name) for name in archived_before}
assert archived_registry_hash==digest(archived_aug/"artifact_hashes.json")
validation={"run_id":RUN_ID,"overall_status":"PASS" if source_ok else "FAIL_SOURCE_CONSISTENCY",
 "training_ready":bool(source_ok),"approval_scope":"retrospective research preprocessing only; no model training executed","variant_checks":variant_checks,
 "checks":{
   "all_augmentation_parents_training_only":"PASS","archived_v1_augmentation_outputs_unchanged":"PASS","approved_v2_consumer_replay":"PASS" if source_ok else "BLOCKED","no_neighbors_or_interpolation":"PASS",
   "original_splits_and_provenance_unchanged":"PASS","original_scalers_unchanged_not_refitted":"PASS",
   "validation_test_arrays_unchanged":"PASS","no_target_attributes_added_to_X":"PASS",
   "synthetic_feature_examples_labeled_as_synthetic":"PASS","no_new_earthquake_IDs_or_timestamps":"PASS",
   "deterministic_saved_plan_replay":"PASS","paired_variant_design":"PASS",
   "noise_labels_and_unselected_features_unchanged":"PASS","same_historical_event_same_epoch_noise":"PASS",
   "coordinates_in_physical_bounds":"PASS","consumer_rejects_invalid_source":"PASS" if not source_ok and consumer_blocked else "NOT_NEEDED",
   "source_rate_semantics":"PASS" if not rate_bad.any() else "FAIL",
   "source_target_gap_units":"PASS" if not gap_bad.any() else "FAIL",
   "operational_feature_availability":"UNKNOWN"},
 "accepted_interpolated_examples":0,
 "accepted_synthetic_feature_examples":int(2*N) if source_ok else 0,
 "diagnostic_synthetic_feature_examples":int(2*N),
 "source_artifacts_before_sha256":protected_before,"source_artifacts_after_sha256":protected_after,
 "source_tensor_sha256":original_tensor_hashes,
 "original_CSV_sha256_unchanged":csv_hash_matches,
 "full_SMOGN_implemented":False,"weighted_sampling_fallback_implemented":True,
 "training_runs":0,"optuna_runs":0,"waveform_files_loaded":0,
 "source_audit":"source_feature_audit.json",
 "interpretation":"Sampling/noise implementation checks are distinct from validity of inherited source features."}
write_json(OUT/"validation_results.json",validation)
# One essential plot: training target distribution only, with acceptance status explicit.
fig,ax=plt.subplots(figsize=(8,4))
positions=np.arange(len(bin_names));width=.19
for offset,row in enumerate(comparison):
    ax.bar(positions+(offset-1.5)*width,
           [row["magnitude_bin_counts"][b]/N for b in bin_names],width,label=row["variant"])
ax.set_xticks(positions,bin_names);ax.set_ylabel("Fraction of training draws")
ax.set_xlabel("Target magnitude (reported ML)")
ax.set_title("Diagnostic previews — source validation failed" if not source_ok else "Training-only variant distributions")
ax.legend(title="Variant");fig.tight_layout()
fig.savefig(OUT/"training_distribution_comparison.png",dpi=150);plt.close(fig)
print("Leakage/replay/immutability checks passed; source acceptance:",validation["overall_status"])


Leakage/replay/immutability checks passed; source acceptance: PASS


In [5]:
research="""# Research-method adaptation

[Chawla et al. (2002), SMOTE](https://doi.org/10.1613/jair.953), Section 4.2 [accessible method](https://www.cs.cmu.edu/afs/cs/project/jair/pub/volume16/chawla02a-html/node6.html), was read. It generates minority-class feature vectors by interpolating selected nearest neighbors. Classification SMOTE does not establish a valid regression target or coherent earthquake-history interpolation here.

[Branco et al. (2017), SMOGN](https://proceedings.mlr.press/v74/branco17a/branco17a.pdf), Sections 2–4 and Algorithm 1, were read. It uses regression relevance to identify important target ranges and combines under-sampling with interpolation for sufficiently close neighbors and Gaussian perturbation otherwise. Its distance-based rule and target construction are not implemented here.

Adopted concept: emphasize relevant high regression targets conservatively. Our weight is 1+0.5×clip((ML−4)/1.5,0,1), using training labels only. Weighted replacement draws retain complete original windows and targets. Sorted parent indices preserve their temporal order. This is the user-authorized fallback, not SMOGN itself. No neighbor graph, flattened sequence interpolation, under-sampling stage or synthetic target magnitude is used.

Our coordinate-only Gaussian noise (standardized sigma 0.01, seed 42) is a separate adaptation. It leaves temporal/rolling relationships and labels unchanged and applies identical event-level perturbations across overlapping windows within an epoch. Noise is not calibrated geophysical uncertainty. Synthetic feature perturbations never become real catalog events.

All four variants keep the same epoch size and original validation/test arrays. B/D share the noise policy; C/D share draws. Future comparison must fix architecture, AdamW optimizer, compute budget and validation-based selection; use test once after selection. Report regression MAE/RMSE/R², high-magnitude MAE and underprediction bias, plus separate classification PR-AUC/precision/recall/F1. No benefit is assumed. Weighted sampling changes the training objective and may worsen overall calibration or MAE.

Independent source consistency checks are mandatory. An inherited feature error is not cured by repetition or noise; previews remain diagnostic until a separately authorized source correction passes validation.
"""
(OUT/"research_adaptation_notes.md").write_text(research,encoding="utf-8")
table=[]
for r in comparison:
    table.append(f"| {r['variant']} | {r['training_draws']} | {r['unique_parent_windows']} | {r['repeat_draws']} | {r['synthetic_feature_examples']} | {100*r['M_ge_4_5_fraction']:.2f}% |")
approval="APPROVED" if source_ok else "BLOCKED"
report=f"""# Phase 1C: corrected-data augmentation revalidation
**{approval} for retrospective research preprocessing. No training performed.**

## Corrected source
Only dataset/v2_timestamp_fixed/bilstm_catalog_seq20.pt supplies numerical model data. All source artifact hashes match the corrected registry. Original v1 dataset and augmentation artifacts are preserved.

The source preprocessing now explicitly converts datetime integers to nanoseconds before rates and intervals. This notebook independently checks corrected training features using time-index rolling windows and timedelta arithmetic. Rate mismatches: {int(rate_bad.sum())}; target-gap mismatches: {int(gap_bad.sum())}; other feature violations: {sum(column_failures.values())}. Actual/saved target-gap ratio: {source_audit['gap_expected_divided_by_saved_min']:g} to {source_audit['gap_expected_divided_by_saved_max']:g}.

## Prepared variants
A clean; B coordinate-only Gaussian noise; C bounded regression-relevance weighted complete-window sampling; D the same weighted draws with the same noise. Seed 42; sigma=0.01 standardized units, latitude/longitude only. No ordinary SMOTE interpolation; weighted sampling is not full SMOGN. No new target magnitudes, IDs or timestamps.

| Variant | Draws | Unique parents | Repeats | Synthetic feature views | ML>=4.5 |
|---|---:|---:|---:|---:|---:|
{chr(10).join(table)}

Each shape is ({N},20,10). C/D have identical parents; B/D apply identical per-event noise within an epoch. No 50/50 quota. Repeated draws and synthetic feature views are explicitly marked in parent_provenance.csv and are not new independent earthquake records.

## Acceptance checks
Source feature semantics, shape/finite constraints, training-only parents, coordinate bounds, unchanged target labels and non-coordinate features, chronological parent/window order, deterministic replay, preserved timestamps and split membership all pass when approval is APPROVED. The saved training consumer was exercised against its deterministic preview. Validation/test tensors and fitted scalers are unchanged and never augmented/refitted here. The v1 augmentation output registry also remains unchanged.

The reproducible plan plus notebook functions materializes accepted epochs on demand; no redundant full augmented tensor files are needed. Sampling/noise config, content hashes, parent records, distribution tables, independent source audit and validation results are saved beside this report.

Operational catalog availability, magnitude calibration and real-world EEW relevance remain unverified. There is no claim that augmentation improves model performance. Training, optimization and fusion are outside this task.

Notebook: ../../phase1b_imbalance_noise_preprocessing.ipynb.
Corrected dataset report: ../../dataset/v2_timestamp_fixed/reports/catalog_preprocessing_report.md.
"""
(OUT/"final_report.md").write_text(report,encoding="utf-8")
output_names=["augmentation_config.json","sampler_manifest.json","sampling_plan.npz","parent_provenance.csv",
 "distribution_comparison.json","distribution_comparison.csv","validation_results.json","source_feature_audit.json",
 "training_distribution_comparison.png","research_adaptation_notes.md","final_report.md"]
write_json(OUT/"artifact_hashes.json",{"run_id":RUN_ID,"training_ready":bool(source_ok),
 "outputs":{name:digest(OUT/name) for name in output_names},
 "hash_convention":"File SHA-256; preview X/target hashes include dtype/shape and numerical bytes.",
 "original_dataset_sha256":protected_before["bilstm_catalog_seq20.pt"]})
def prepend_current(path,body):
    start="<!-- PHASE1C_CURRENT_START -->";end="<!-- PHASE1C_CURRENT_END -->"
    old=path.read_text(encoding="utf-8") if path.exists() else ""
    if start in old and end in old:old=old.split(end,1)[1].lstrip()
    path.parent.mkdir(exist_ok=True)
    path.write_text(start+"\n"+body.strip()+"\n"+end+"\n\n"+old,encoding="utf-8")
summary=f"""# Phase 1C - timestamp-fixed v2 {approval} (2026-11-09)
- Explicit ns conversion replaces native-unit integer arithmetic. Rates, target gaps, dependency intervals and time-derived summaries rebuilt from original CSV; no tensor rescaling shortcut.
- Corrected dataset: dataset/v2_timestamp_fixed/. New train-only scalers; previous artifacts unchanged. Targets, schema, event identities/timestamps and chronological splits preserved.
- Shapes train/val/test: (4973,20,10)/(1065,20,10)/(1067,20,10). Independent known-time/boundary and full-catalog checks passed.
- Corrected augmentation: augmentation_experiment/v2_timestamp_fixed/. Four variants, seed 42, coordinate sigma 0.01; complete-window weighting, no interpolation.
- ML>=4.5 proportions: A/B {100*comparison[0]['M_ge_4_5_fraction']:.2f}%; C/D {100*comparison[2]['M_ge_4_5_fraction']:.2f}%.
- Source feature rate/gap violations now {int(rate_bad.sum())}/{int(gap_bad.sum())}. Parent, label, split, scaler and validation/test preservation checks pass.
- Read dataset/v2_timestamp_fixed/reports/catalog_preprocessing_report.md and augmentation_experiment/v2_timestamp_fixed/final_report.md.
- Both corrected notebooks executed. No training/Optuna/fusion performed. Operational availability remains UNKNOWN; approval is for retrospective research preprocessing only.
- Earlier v1 notes below are archived; use v2 exclusively for future authorized work.

## Archived notes"""
prepend_current(ROOT/"project_status.md",summary)
prepend_current(ROOT/".agent-memory/checkpoint.md",summary)
print("Phase 1C dataset and augmentation:",approval,"; no training executed.")


Phase 1C dataset and augmentation: APPROVED ; no training executed.
